In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from kneed import KneeLocator
from scipy.linalg import eig

# Reproducible project paths and output locations

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from inside the DELVE repository.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))









from project_utils import TABLES_DIR, configure_plots, save_figure, ensure_output_dirs, FIGURES_DIR
from functions import LG_sym, calc_differential_vec, diffusion_map

configure_plots()
ensure_output_dirs()

In [ ]:
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
plt.rcParams["mathtext.it"] = "Times New Roman:italic"
plt.rcParams["mathtext.bf"] = "Times New Roman:bold"

In [ ]:
# Ensure imports work whether Jupyter runs from project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Line vs. Triangle

In [ ]:
n = 5000
N=2000
W = 4
L = 0.5
X1 = np.zeros((n,1))
X2 = np.zeros((n,2))


w_ = np.random.uniform(0,W,n)
l_ = np.random.uniform(0,L,n)
#Phi = np.random.uniform(0,1,n)

X1 = w_[np.newaxis].T
X2 = np.vstack((w_, l_)).T
non_trash = np.where(X2[:,1]<=(L - (L/W)*X2[:,0]))[0]
X2 = X2[non_trash,:]
X2 = X2[:N,:]

X1 = X1[non_trash,:]
X1 = X1[:N,:]


w = X2[:,0]
l = X2[:,1]

In [ ]:
plt.scatter(X2[:,0], X2[:,1], s = 0.75)
save_figure("rectangle", dpi = 400)
plt.show()

plt.scatter(X1, np.tile(([0]),N), s = 0.5)
save_figure("line", dpi = 400)
plt.show()

In [ ]:
# Core operators/eigenvectors for all methods on the representative sample
P1, Q1, K1 = diffusion_map(X1, adaptive=800, coifman_lafon = False)
P2, Q2, K2 = diffusion_map(X2, adaptive=800, coifman_lafon = False)

L1, d1, v1 = LG_sym(K1)
L2, d2, v2 = LG_sym(K2)

kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.5)
tau1 = kl.knee
kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.5)
tau2 = kl.knee
print(tau1)
print(tau2)

_, u1 = calc_differential_vec(L2, v1, tau1)
_, u2 = calc_differential_vec(L1, v2, tau2)

S = P2 @ Q1 + P1 @ Q2
D = P2 @ Q1 - P1 @ Q2

_, _ = eig(S)  # computed in original workflow; not used directly in final outputs
ea_vals, va = eig(D)
VA_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
VA_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])

g1 = np.diag(np.sum(K1, axis=1)) - K1
g2 = np.diag(np.sum(K2, axis=1)) - K2

m1 = g1 + 1e-6 * np.eye(g1.shape[0])
m2 = g2 + 1e-6 * np.eye(g2.shape[0])

fk1 = np.linalg.inv(m1 + m2) @ m1
fk2 = np.linalg.inv(m1 + m2) @ m2

fk_vals_1, eig_vec_fk_1 = eig(fk1)
fk_vals_2, eig_vec_fk_2 = eig(fk2)

eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_vals_1)[::-1]]
eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_vals_2)[::-1]]

## DELVE

In [ ]:
plt.figure(figsize=(5, 4))

plt.scatter(
    X2[:, 0],
    X2[:, 1],
    c=u1[:, 0],
    cmap="PiYG",
    s=18,
    alpha=0.5,
)

plt.xlabel(r"$\theta$", fontsize=18)
plt.ylabel(
    r"$\psi^B$",
    rotation=0,
    fontsize=18,
    labelpad=20,
)

# Remove ticks
plt.xticks([])
plt.yticks([])

save_figure(
    "triangle_colored.pdf",
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.05,
)

plt.show()

In [ ]:
print(abs(spearmanr(l, u1[:, 0]).statistic))

### FKT

In [ ]:
print(abs(spearmanr(l, eig_vec_fk_1[:, 0]).statistic))

## Shnitzer et. al

In [ ]:
print(abs(spearmanr(l, VA_imag[:, 0]).statistic))
print(abs(spearmanr(l, VA_real[:, 0]).statistic))

### Now N times for different L values:

In [ ]:
def triangle_corr(W, L, N, K = 200):
    
    n = round(2.5*N)
    X1 = np.zeros((n,1))
    X2 = np.zeros((n,2))


    w_ = np.random.uniform(0,W,n)
    l_ = np.random.uniform(0,L,n)

    X1 = w_[np.newaxis].T
    X2 = np.vstack((w_, l_)).T
    
    non_trash = np.where(X2[:,1]<=(L - (L/W)*X2[:,0]))[0]
    
    X2 = X2[non_trash,:]
    X2 = X2[:N,:]
    X1 = X1[non_trash,:]
    X1 = X1[:N,:]

    w = X2[:,0]
    l = X2[:,1] 
    
    
    # Core operators/eigenvectors for all methods on the representative sample
    P1, Q1, K1 = diffusion_map(X1, adaptive=K, coifman_lafon = False)
    P2, Q2, K2 = diffusion_map(X2, adaptive=K, coifman_lafon = False)
    
    L1, d1, v1 = LG_sym(K1)
    L2, d2, v2 = LG_sym(K2)
    
    kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing")
    tau1 = kl.knee
    kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing")
    tau2 = kl.knee
    # print(tau1)
    # print(tau2)
    
    _, u1 = calc_differential_vec(L2, v1, tau1)
    _, u2 = calc_differential_vec(L1, v2, tau2)
    
    S = P2 @ Q1 + P1 @ Q2
    D = P2 @ Q1 - P1 @ Q2
    
    _, _ = eig(S)  # computed in original workflow; not used directly in final outputs
    ea_vals, va = eig(D)
    VA_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
    VA_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])
    
    g1 = np.diag(np.sum(K1, axis=1)) - K1
    g2 = np.diag(np.sum(K2, axis=1)) - K2
    
    m1 = g1 + 1e-6 * np.eye(g1.shape[0])
    m2 = g2 + 1e-6 * np.eye(g2.shape[0])
    
    fk1 = np.linalg.inv(m1 + m2) @ m1
    fk2 = np.linalg.inv(m1 + m2) @ m2
    
    fk_vals_1, eig_vec_fk_1 = eig(fk1)
    fk_vals_2, eig_vec_fk_2 = eig(fk2)
    
    eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_vals_1)[::-1]]
    eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_vals_2)[::-1]]

 
    #calc corr for DV 
    corr_DV = abs(spearmanr(l, u1[:, 0]).statistic)
    
    #calc corr for FK
    corr_FK = abs(spearmanr(l,eig_vec_fk_1[:,0]).statistic) 

    corr_sh_imag = abs(spearmanr(l,VA_imag[:,0]).statistic)
    corr_sh_real = abs(spearmanr(l,VA_real[:,0]).statistic)
    
    return corr_DV, corr_FK, corr_sh_imag, corr_sh_real

In [ ]:
B = 50
L_list = np.round(np.linspace(0.5,16,8),2)
N = len(L_list)
acc_triangle = np.zeros((B,N,4))

for j in range(B):
    for i in range(N):
        n = 2000
        W = 4
        L = L_list[i]
        acc_triangle[j,i,:] = triangle_corr(W,L, n, K = 800)
        print("N: "+ str(i))
    print("B: "+ str(j))

mean_acc_triangle = np.mean(acc_triangle,axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

methods = [
    ("DELVE", 0),
    ("FKT", 1),
    ("Shnitzer + (real)", 2),
    ("Shnitzer + (imag)", 3),
]

for label, i in methods:
    mean = 100 * mean_acc_triangle[:, i]
    std = 100 * std_acc_triangle[:, i]

    line, = ax.plot(
        L_list,
        mean,
        linewidth=2,
        label=label,
    )

    ax.fill_between(
        L_list,
        mean - std,
        mean + std,
        alpha=0.2,
        color=line.get_color(),
    )

# Labels
ax.set_xlabel(r"$L$", fontsize=18)
ax.set_ylabel(r"Correlation with $\ell$ [%]", fontsize=18)

# Ticks
ax.set_xticks(np.round(L_list, 1))
ax.tick_params(axis="both", labelsize=16)

# Grid
ax.grid(alpha=0.25)

# Legend
ax.legend(fontsize=13, loc="best", frameon=True)

fig.tight_layout()

fig.savefig(FIGURES_DIR / "triangle_over_L.pdf",
    dpi=400,
    bbox_inches="tight",
    pad_inches=0.05,
)

plt.show()

In [ ]:
columns = ['DV', 'FKT', 'Shnitzer et al. (real)', 'Shnitzer et al. (imag)']
df = pd.DataFrame(np.abs(acc_triangle), columns=columns)

# Keep original paper-summary slicing behavior for compatibility
SUMMARY_ROWS = 342
display_cols = ['DV', 'Shnitzer et al. (real)', 'Shnitzer et al. (imag)', 'FKT']
display_names = ['DELVE', 'Shnitzer et al. (real)', 'Shnitzer et al. (imag)', 'FK']

summary_values = []
for col in display_cols:
    mean_val = df.iloc[:SUMMARY_ROWS][col].mean()
    sd_val = df.iloc[:SUMMARY_ROWS][col].std()
    summary_values.append(f'{mean_val:.3f} ({sd_val:.3f})')

summary_df = pd.DataFrame([summary_values], columns=display_names, index=['Mean (SD)'])
display(summary_df)
summary_df.to_latex(TABLES_DIR / "Tri_vsLine_summary_table.tex", index=True, float_format='%.3f')